In [1]:
# from google.colab import drive
# drive.mount('/content/drive')

In [2]:
# %%capture --no-stderr
# !pip install python-dotenv langchain-community langchain-core langchain langchain-openai langchain-chroma

In [3]:
# 환경변수 설정

In [4]:
# 라이브러리 불러오기
from dotenv import load_dotenv
import os
from langchain_openai import ChatOpenAI

In [5]:
# .env 파일에서 환경 변수 로드 (.env 파일에는 OPENAI API 키값을 적으면 됩니다. -> OPENAI_API_KEY=...)
load_dotenv("/content/.env")
# 환경 변수에서 API 키 가져오기
api_key = os.getenv("OPENAI_API_KEY")
# 오픈AI 대규모 언어 모델 초기화
llm = ChatOpenAI()

In [6]:
# <이전 대화를 포함한 메시지 전달>

# 라이브러리 불러오기
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI

chat = ChatOpenAI(model="gpt-4o-mini")

# 프롬프트 템플릿 정의: 금융 상담 역할
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 사용자에게 최선의 금융 조언을 제공합니다."),
        ("placeholder", "{messages}"),  # 대화 이력 추가
    ]
)

# 프롬프트와 모델을 연결하여 체인 생성
chain = prompt | chat

# 이전 대화를 포함한 메시지 전달
ai_msg = chain.invoke(
    {
        "messages": [
            ("human", "저축을 늘리기 위해 무엇을 할 수 있나요?"),  # 사용자의 첫 질문
            ("ai", "저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요."),  # 챗봇의 답변
            ("human", "방금 뭐라고 했나요?"),  # 사용자의 재확인 질문
        ],
    }
)
print(ai_msg.content)  # 챗봇의 응답 출력


저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하는 것을 추천드립니다. 이렇게 하면 저축을 더 체계적으로 관리할 수 있습니다. 추가적으로 다음과 같은 방법도 고려해 보세요:

1. **예산 세우기**: 수입과 지출을 명확히 파악하여 불필요한 지출을 줄이세요.
2. **비상금 마련**: 예상치 못한 지출을 대비해 비상금을 저축하세요.
3. **저축 계좌 활용**: 높은 이율을 제공하는 저축 계좌를 선택해 이자를 최대한 활용하세요.
4. **소비 습관 점검**: 필요 없는 지출을 줄이고, 할인이나 쿠폰을 활용해 비용을 절감하세요.
5. **목표 설정**: 저축 목표를 구체적으로 설정하고, 달성할 때마다 자신을 보상해 주세요.

이러한 방법들을 통해 저축을 늘릴 수 있습니다.


In [7]:
# <`ChatMessageHistory`를 사용한 메시지 관리>

from langchain_community.chat_message_histories import ChatMessageHistory

# 대화 이력 저장을 위한 클래스 초기화
chat_history = ChatMessageHistory()

# 사용자 메시지 추가
chat_history.add_user_message("저축을 늘리기 위해 무엇을 할 수 있나요?")
chat_history.add_ai_message("저축 목표를 설정하고, 매달 자동 이체로 일정 금액을 저축하세요.")

# 새로운 질문 추가 후 다시 체인 실행
chat_history.add_user_message("방금 뭐라고 했나요?")
ai_response = chain.invoke({"messages": chat_history.messages})
print(ai_response.content)  # 챗봇은 이전 메시지를 기억하여 답변합니다.

저축을 늘리기 위해 저축 목표를 설정하고, 매달 자동 이체를 통해 일정 금액을 저축하는 것이 좋다고 말씀드렸습니다. 이렇게 하면 계획적으로 저축을 할 수 있습니다. 추가적인 조언이 필요하시면 말씀해 주세요!


In [8]:
# <` RunnableWithMessageHistory`를 사용한 메시지 관리>

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_community.chat_message_histories import ChatMessageHistory

# 시스템 메시지와 대화 이력을 사용하는 프롬프트 템플릿 정의
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "당신은 금융 상담사입니다. 모든 질문에 최선을 다해 답변하십시오."),
        ("placeholder", "{chat_history}"),  # 이전 대화 이력
        ("human", "{input}"),  # 사용자의 새로운 질문
    ]
)

# 대화 이력을 관리할 체인 설정
chat_history = ChatMessageHistory()
chain = prompt | chat

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# RunnableWithMessageHistory 클래스를 사용해 체인을 감쌉니다
chain_with_message_history = RunnableWithMessageHistory(
    chain,
    lambda session_id: chat_history,  # 세션 ID에 따라 대화 이력을 불러오는 함수
    input_messages_key="input",  # 입력 메시지의 키 설정
    history_messages_key="chat_history",  # 대화 이력의 키 설정
)

# 질문 메시지 체인 실행
chain_with_message_history.invoke(
    {"input": "저축을 늘리기 위해 무엇을 할 수 있나요?"},
    {"configurable": {"session_id": "unused"}},
).content


'저축을 늘리기 위해 다음과 같은 방법을 고려해보세요:\n\n1. **예산 작성**: 월별 수입과 지출을 정리하여 예산을 세우세요. 불필요한 지출을 줄이고, 저축할 수 있는 금액을 명확히 파악할 수 있습니다.\n\n2. **자동 저축**: 급여가 입금되면 자동으로 저축 계좌로 일정 금액을 이체하도록 설정하세요. 이렇게 하면 저축을 더 쉽게 일관되게 할 수 있습니다.\n\n3. **비상금 마련**: 먼저 비상금을 마련하세요. 예상치 못한 지출을 대비해 3~6개월치 생활비를 저축하는 것이 좋습니다.\n\n4. **지출 습관 점검**: 필요 없는 구독 서비스나 사용하지 않는 멤버십을 취소하고, 쇼핑할 때는 세일이나 할인 쿠폰을 활용하세요.\n\n5. **목표 설정**: 구체적인 저축 목표를 설정하세요. 예를 들어, 여행, 집 구매, 은퇴 자금 등 목표를 명확히 하면 저축 동기가 강화됩니다.\n\n6. **추가 수입 창출**: 부업이나 프리랜스 일을 통해 추가 수입을 얻어 저축에 보태세요.\n\n7. **금융 상품 활용**: 고금리 저축 계좌나 적금, 펀드 등을 활용하여 저축의 이자를 최대화하세요.\n\n8. **소비 패턴 분석**: 월별 지출 내역을 분석하여 어떤 항목에서 절약할 수 있는지 찾아보세요. \n\n이러한 방법들을 통해 저축을 늘리고 재정적인 목표를 달성하는 데 도움이 될 것입니다.'

In [9]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_message_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content


'당신은 "저축을 늘리기 위해 무엇을 할 수 있나요?"라고 질문하셨습니다. 저축을 늘리기 위한 다양한 방법에 대해 설명해 드렸습니다. 추가 질문이나 궁금한 사항이 있으시면 알려주세요!'

In [10]:
# <메시지 트리밍 예제>

# 라이브러리 불러오기
from langchain_core.messages import trim_messages
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter

# 메시지 트리밍 유틸리티 설정
trimmer = trim_messages(strategy="last", max_tokens=2, token_counter=len)

# 트리밍된 대화 이력과 함께 체인 실행
chain_with_trimming = (
    RunnablePassthrough.assign(chat_history=itemgetter("chat_history") | trimmer)
    | prompt
    | chat
)

# 트리밍된 대화 이력을 사용하는 체인 설정
chain_with_trimmed_history = RunnableWithMessageHistory(
    chain_with_trimming,
    lambda session_id: chat_history,
    input_messages_key="input",
    history_messages_key="chat_history",
)

# 새로운 대화 내용 추가 후 체인 실행
chain_with_trimmed_history.invoke(
    {"input": "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
)



ImportError: cannot import name 'trim_messages' from 'langchain_core.messages' (d:\WorkSpace\Python\langchain-tutorial\Ch01. Langchain Basics\venv\lib\site-packages\langchain_core\messages\__init__.py)

In [ ]:
# 새로운 입력 메시지를 추가하고 체인을 실행
chain_with_trimmed_history.invoke(
    {"input": "내가 방금 뭐라고 했나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "finance_session_1"}}  # 세션 ID 설정
).content


'당신은 "저는 5년 내에 집을 사기 위해 어떤 재정 계획을 세워야 하나요?"라고 질문하셨습니다. 이 질문에 대해 집 구매를 위한 재정 계획을 세우는 방법에 대해 여러 가지 단계와 조언을 드렸습니다. 추가적인 질문이나 더 알고 싶은 내용이 있으면 말씀해 주세요!'

In [ ]:
# <이전 대화 요약 내용 기반으로 답변하기>

def summarize_messages(chain_input):
    stored_messages = chat_history.messages
    if len(stored_messages) == 0:
        return False
    # 대화를 요약하기 위한 프롬프트 템플릿 설정
    summarization_prompt = ChatPromptTemplate.from_messages(
        [
            ("placeholder", "{chat_history}"),  # 이전 대화 이력
            (
                "user",
                "이전 대화를 요약해 주세요. 가능한 한 많은 세부 정보를 포함하십시오.",  # 요약 요청 메시지
            ),
        ]
    )

    # 요약 체인 생성 및 실행
    summarization_chain = summarization_prompt | chat
    summary_message = summarization_chain.invoke({"chat_history": stored_messages})
    chat_history.clear()  # 요약 후 이전 대화 삭제
    chat_history.add_message(summary_message)  # 요약된 메시지를 대화 이력에 추가
    return True

In [ ]:
# 대화 요약을 처리하는 체인 설정
chain_with_summarization = (
    RunnablePassthrough.assign(messages_summarized=summarize_messages)
    | chain_with_message_history
)

# 요약된 대화를 기반으로 새로운 질문에 응답
print(chain_with_summarization.invoke(
    {"input": "저에게 어떤 재정적 조언을 해주셨나요?"},  # 사용자의 질문
    {"configurable": {"session_id": "unused"}}  # 세션 ID 설정
).content)


재정적 조언을 드리기 위해 아래의 포인터들을 정리해드리겠습니다. 이러한 조언들은 일반적인 재정 관리 원칙에 기반합니다:

1. **예산 수립**:
   - 월별 수입과 지출을 기록하고 예산을 세워 체계적으로 관리합니다. 필요지출과 선택지출을 구분해 불필요한 지출을 줄이고 재정적으로 건강한 상태를 유지하도록 합니다.

2. **저축 습관 형성**:
   - 월급 이체일에 저축 계좌로 자동 이체를 설정하여 항상 일정한 금액을 저축하게 됩니다.
   - 목표를 설정하여 저축을 목표지향적으로 이끌어갑니다.

3. **부채 관리**:
   - 고금리 부채를 우선적으로 갚고, 신용 점수를 꾸준히 체크하여 재정 상태를 점검합니다.
   - 필요한 경우 대출 조건이나 상환 계획을 재조정합니다.

4. **비상금 마련**:
   - 최소 3~6개월치 생계비를 비상금으로 저축합니다. 이는 예기치 못한 상황에서 금융적인 안전망이 됩니다.

5. **투자 고려**:
   - 목돈을 만든 후에는 자산을 증식하기 위해 주식, 채권, 펀드 등 다양한 투자를 고려할 수 있습니다. 자신의 위험 감수 성향에 맞는 투자 전략을 수립합니다.

6. **재정 목표 설정**:
   - 단기 및 장기 목표를 설정합니다. 예를 들어, 5년 내 집 구매를 목표로 하는 경우 정확한 저축액과 구체적인 계획이 필요합니다.

7. **전문가 상담**:
   - 금융 지식이 부족하다면 전문 상담사에게 조언을 구하고 다양한 금융 상품을 비교하여 결정합니다.

8. **주기적인 점검**:
   - 자신의 재정 계획을 정기적으로 점검하고 필요 시 수정합니다. 생활 변화에 따라 목표를 조정할 수 있습니다.

이런 조언들은 개인의 재정 상황과 목표에 따라 다르게 적용될 수 있습니다. 필요하신 부분이나 더 구체적인 상황에 대한 질문이 있다면 말씀해 주세요!
